In [2]:
import os
import cv2
import random
import numpy as np
import xml.etree.ElementTree as ET
import albumentations as A
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict


C:\Users\Asus\anaconda3\envs\TorchGPU\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Asus\anaconda3\envs\TorchGPU\lib\site-packages\albumentations\check_version.py:147: UserWarning: Error fetching version info <urlopen error _ssl.c:1000: The handshake operation timed out>
  data = fetch_version_info()


In [ ]:

random.seed(42)
np.random.seed(42)

BASE_PATH      = r'C:\Users\Gaurav\Documents\AIDSML\ManufacturingProject\Manufacturing Surface Anomaly Inspector'
IMAGES_DIR     = os.path.join(BASE_PATH, 'images')
LABELS_DIR     = os.path.join(BASE_PATH, 'labels')
AUG_IMAGES_DIR = os.path.join(BASE_PATH, 'augmented_images')
AUG_LABELS_DIR = os.path.join(BASE_PATH, 'augmented_labels')

TARGET         = 600  
RANDOM_SEED    = 42

os.makedirs(AUG_IMAGES_DIR, exist_ok=True)
os.makedirs(AUG_LABELS_DIR, exist_ok=True)

def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size   = root.find('size')
    width  = int(size.find('width').text)
    height = int(size.find('height').text)
    objects = []
    for obj in root.findall('object'):
        name   = obj.find('name').text
        bndbox = obj.find('bndbox')
        objects.append({
            'name': name,
            'xmin': float(bndbox.find('xmin').text),
            'ymin': float(bndbox.find('ymin').text),
            'xmax': float(bndbox.find('xmax').text),
            'ymax': float(bndbox.find('ymax').text),
        })
    return width, height, objects

    
def write_xml(out_path, filename, width, height, objects):
    annotation = ET.Element('annotation')
    ET.SubElement(annotation, 'filename').text = filename
    size_elem = ET.SubElement(annotation, 'size')
    ET.SubElement(size_elem, 'width').text  = str(width)
    ET.SubElement(size_elem, 'height').text = str(height)
    ET.SubElement(size_elem, 'depth').text  = '3'
    for obj in objects:
        obj_elem = ET.SubElement(annotation, 'object')
        ET.SubElement(obj_elem, 'name').text      = obj['name']
        ET.SubElement(obj_elem, 'pose').text      = 'Unspecified'
        ET.SubElement(obj_elem, 'truncated').text = '0'
        ET.SubElement(obj_elem, 'difficult').text = '0'
        bndbox = ET.SubElement(obj_elem, 'bndbox')
        ET.SubElement(bndbox, 'xmin').text = str(int(obj['xmin']))
        ET.SubElement(bndbox, 'ymin').text = str(int(obj['ymin']))
        ET.SubElement(bndbox, 'xmax').text = str(int(obj['xmax']))
        ET.SubElement(bndbox, 'ymax').text = str(int(obj['ymax']))
    tree = ET.ElementTree(annotation)
    ET.indent(tree, space='  ')
    tree.write(out_path, encoding='utf-8', xml_declaration=True)


def build_transform(h, w):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.RandomResizedCrop(size=(h, w), scale=(0.75, 1.0), ratio=(0.9, 1.1), p=0.4),
        A.Perspective(scale=(0.02, 0.06), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.4),
        A.CLAHE(clip_limit=3.0, p=0.3),
        A.Sharpen(alpha=(0.1, 0.4), p=0.3),
    ],
    bbox_params=A.BboxParams(
        format='pascal_voc',
        label_fields=['class_labels'],
        min_visibility=0.3,
    ))

# STEP 1: Count original bbox instances per class 
print("Counting original bbox instances per class...")
bbox_counts = defaultdict(int)

for xml_file in os.listdir(LABELS_DIR):
    if not xml_file.endswith('.xml'):
        continue
    _, _, objects = parse_xml(os.path.join(LABELS_DIR, xml_file))
    for obj in objects:
        bbox_counts[obj['name']] += 1

print(f"\n{'Class':<15} {'Current':>8} {'Target':>8} {'Need':>8} {'Status':>10}")
print("-" * 55)
classes_over  = set()   #already fulfilled target
budget        = {}      #BB per class needed

for cls, count in sorted(bbox_counts.items()):
    need   = max(0, TARGET - count)
    status = 'skip' if need == 0 else f'+{need}'
    print(f"{cls:<15} {count:>8} {TARGET:>8} {need:>8} {status:>10}")
    if count >= TARGET:
        classes_over.add(cls)
    else:
        budget[cls] = need

print(f"\nClasses excluded from augmentation (≥{TARGET} bboxes): {classes_over}")



#  STEP 2: Find eligible source images 
print("\nFinding eligible source images...")

class_to_images = defaultdict(list)   # class → list of eligible image filenames
image_objects   = {}                  # cache parsed objects per xml stem

for xml_file in os.listdir(LABELS_DIR):
    if not xml_file.endswith('.xml'):
        continue
    xml_path = os.path.join(LABELS_DIR, xml_file)
    _, _, objects = parse_xml(xml_path)
    names_in_image = {obj['name'] for obj in objects}

    if names_in_image & classes_over:
        continue

    stem = Path(xml_file).stem
    image_objects[stem] = objects

    for cls in names_in_image:
        if cls in budget:
            class_to_images[cls].append(stem)

print(f"Eligible source images: {len(image_objects)}")
for cls, imgs in sorted(class_to_images.items()):
    print(f"   {cls:<15} → {len(imgs)} eligible images")



# STEP 3: Build augmentation plan (bbox-budget aware) 
# Round-robin per class, decrement bbox budget for ALL classes in the image
print("\nBuilding augmentation plan...")

aug_plan      = defaultdict(int)   # stem → how many augmented copies to make
live_budget   = dict(budget)       # mutable copy

# Process rarest class first
for cls in sorted(live_budget, key=lambda c: bbox_counts[c]):
    if live_budget[cls] <= 0:
        continue
    candidates = class_to_images[cls][:]
    if not candidates:
        print(f"No eligible images for {cls} — skipping")
        continue
    random.shuffle(candidates)
    idx = 0
    while live_budget[cls] > 0:
        stem    = candidates[idx % len(candidates)]
        objects = image_objects[stem]
        aug_plan[stem] += 1

        # Decrement budget for ALL classes present in this image
        for obj in objects:
            c = obj['name']
            if c in live_budget:
                live_budget[c] = max(0, live_budget[c] - 1)
        idx += 1

total_new = sum(aug_plan.values())
print(f"Plan ready — {len(aug_plan)} unique images to augment, {total_new} new copies total")



#  STEP 4: Execute augmentation 
print(f"\n🔄 Augmenting...\n")
copy_counters = defaultdict(int)
saved = 0
skipped = 0

for stem, n_copies in tqdm(aug_plan.items()):
    # Find image file
    img_file = None
    for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
        candidate = os.path.join(IMAGES_DIR, stem + ext)
        if os.path.exists(candidate):
            img_file = candidate
            break
    if img_file is None:
        skipped += 1
        continue

    image = cv2.imread(img_file)
    if image is None:
        skipped += 1
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w  = image.shape[:2]

    objects = image_objects[stem]
    bboxes  = [[o['xmin'], o['ymin'], o['xmax'], o['ymax']] for o in objects]
    labels  = [o['name'] for o in objects]

    for i in range(n_copies):
        try:
            result = build_transform(h, w)(
                image=image,
                bboxes=bboxes,
                class_labels=labels,
            )
        except Exception as e:
            print(f"Transform failed on {stem} copy {i}: {e}")
            continue

        aug_img    = result['image']
        aug_bboxes = result['bboxes']
        aug_labels = result['class_labels']

        if len(aug_bboxes) == 0:
            continue   # all boxes cropped out

        aug_h, aug_w = aug_img.shape[:2]
        copy_counters[stem] += 1
        out_name = f"{stem}_aug{copy_counters[stem]:03d}_11"
        out_img_path = os.path.join(AUG_IMAGES_DIR, out_name + '.jpg')
        out_xml_path = os.path.join(AUG_LABELS_DIR, out_name + '.xml')

        cv2.imwrite(
            out_img_path,
            cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR),
            [cv2.IMWRITE_JPEG_QUALITY, 95],
        )

        new_objects = [
            {
                'name': cls_name,
                'xmin': max(0, xmin),
                'ymin': max(0, ymin),
                'xmax': min(aug_w, xmax),
                'ymax': min(aug_h, ymax),
            }
            for (xmin, ymin, xmax, ymax), cls_name in zip(aug_bboxes, aug_labels)
        ]
        write_xml(out_xml_path, out_name + '.jpg', aug_w, aug_h, new_objects)
        saved += 1

print(f"\n{'='*55}")
print(f"Augmentation complete!")
print(f"   New images saved : {saved}")
print(f"   Skipped          : {skipped}")
print(f"\nVerify final bbox counts by re-running your counting cell")
print(f"   on BOTH Original_label + augmented_labels combined.")
print(f"{'='*55}")

In [4]:

random.seed(42)

LABELS_DIR  = r'C:\Users\Asus\Documents\AIDSML\ManufacturingProject\Manufacturing Surface Anomaly Inspector\NeededTOAuglabels1'
IMAGES_DIR  = r'C:\Users\Asus\Documents\AIDSML\ManufacturingProject\Manufacturing Surface Anomaly Inspector\NeededTOAugimages1'

TARGETS = {  
    '6_siban':   200,  
}

ACTION = 'delete'   # 'dry_run'

def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size   = root.find('size')
    width  = int(size.find('width').text)
    height = int(size.find('height').text)
    objects = []
    for obj in root.findall('object'):
        name   = obj.find('name').text
        bndbox = obj.find('bndbox')
        objects.append({
            'name': name,
            'xmin': float(bndbox.find('xmin').text),
            'ymin': float(bndbox.find('ymin').text),
            'xmax': float(bndbox.find('xmax').text),
            'ymax': float(bndbox.find('ymax').text),
        })
    return width, height, objects

#  STEP 1: Find images that contain ONLY the target claSS
print("Scanning labels...\n")

candidates = {cls: [] for cls in TARGETS}

for xml_file in os.listdir(LABELS_DIR):
    if not xml_file.endswith('.xml'):
        continue
    xml_path = os.path.join(LABELS_DIR, xml_file)
    _, _, objects = parse_xml(xml_path)
    names_in_image = {obj['name'] for obj in objects}

    for cls in TARGETS:
        if names_in_image == {cls}:
            bbox_count = sum(1 for obj in objects if obj['name'] == cls)
            candidates[cls].append((xml_file, bbox_count))



# STEP 2: Greedily select images until bbox removal target is met 
print(f"{'Class':<15} {'Eligible images':>16} {'Total bboxes':>14}")
print("-" * 50)
for cls, imgs in candidates.items():
    total_bboxes = sum(bc for _, bc in imgs)
    print(f"{cls:<15} {len(imgs):>16} {total_bboxes:>14}")

print()

to_delete = {}   #list of xml filenames selected for deletion

for cls, remove_count in TARGETS.items():
    imgs = candidates[cls][:]
    random.shuffle(imgs)

    selected   = []
    removed_bb = 0

    for xml_file, bbox_count in imgs:
        if removed_bb >= remove_count:
            break
        selected.append((xml_file, bbox_count))
        removed_bb += bbox_count

    to_delete[cls] = selected

    print(f"{'='*55}")
    print(f"  Class          : {cls}")
    print(f"  Target removal : {remove_count} bbox instances")
    print(f"  Images selected: {len(selected)}")
    print(f"  Bboxes removed : {removed_bb}  {'slightly over — nearest whole image' if removed_bb > remove_count else '✅'}")
    if len(selected) <= 10:
        for f, bc in selected:
            print(f"    🗑  {f}  ({bc} bbox{'es' if bc>1 else ''})")
    else:
        for f, bc in selected[:5]:
            print(f"    🗑  {f}  ({bc} bbox{'es' if bc>1 else ''})")
        print(f"    ... and {len(selected)-5} more")



# STEP 3: Execute 
if ACTION == 'dry_run':
    print(f"\n[DRY RUN] Nothing deleted. Set ACTION = 'delete' to proceed.")

elif ACTION == 'delete':
    total_deleted = 0
    for cls, selected in to_delete.items():
        for xml_file, _ in selected:
            stem     = Path(xml_file).stem
            xml_path = os.path.join(LABELS_DIR, xml_file)

            # Delete label
            if os.path.exists(xml_path):
                os.remove(xml_path)

            # Delete matching image
            for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                img_path = os.path.join(IMAGES_DIR, stem + ext)
                if os.path.exists(img_path):
                    os.remove(img_path)
                    break

            total_deleted += 1

    print(f"\nDeleted {total_deleted} image+label pairs.")
    print(f"   Re-run your counting cell to verify final bbox counts.")

🔍 Scanning labels...

Class            Eligible images   Total bboxes
--------------------------------------------------
6_siban                      648            779

  Class          : 6_siban
  Target removal : 200 bbox instances
  Images selected: 176
  Bboxes removed : 203  ⚠️ slightly over — nearest whole image
    🗑  img_01_425005700_00500.xml  (1 bbox)
    🗑  img_07_425243200_00001.xml  (1 bbox)
    🗑  img_02_436184600_00782.xml  (1 bbox)
    🗑  img_06_4406645900_00449.xml  (1 bbox)
    🗑  img_02_436153600_00709.xml  (1 bbox)
    ... and 171 more

✅ Deleted 176 image+label pairs.
   Re-run your counting cell to verify final bbox counts.
